In [1]:
import spacy
from fastcoref import spacy_component
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()
for i, p in enumerate(podcasts):
    print(f"{i}: {p['title']}")

/home/adamj/anaconda3/envs/whspr/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0: Verdict with Ted Cruz
1: Leger om livet
2: Norsken, svensken og dansken
3: Huberman Lab
4: The Ben Shapiro Show
5: The Megyn Kelly Show
6: The Daily
7: Checks and Balance from The Economist
8: Pod Save America
9: Logbuch:Netzpolitik
10: Apokalypse & Filterkaffee
11: Lage der Nation - der Politik-Podcast aus Berlin
12: Inside Europe | Deutsche Welle
13: Forklart
14: Oppdatert
15: USApodden
16: Det Store Bildet
17: Best of the Left - Progressive Politics and Culture, Curated by Humans, Not Algorithms
18: Gaslit Nation with Andrea Chalupa and Sarah Kendzior
19: The New Abnormal
20: The Fox News Rundown
21: The Dan Bongino Show
22: Aftenpodden USA
23: Leading
24: The Rest Is Politics
25: Dagens Eko
26: The Doctor's Farmacy with Mark Hyman, M.D.
27: Ancient Health Podcast
28: ZOE Science & Nutrition
29: Stetoskopet – Tidsskriftets podkast
30: Just One Thing - with Michael Mosley
31: Genstart


In [7]:
import spacy
from fastcoref import spacy_component
import requests

for podcast in podcasts:
    if not podcast["language"].startswith("en"):
        #print(f"skipping {podcast['title']}, language is {podcast['language']}")
        continue
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()
                    utterances = seg["utterance_set"]
                    # check if coref already exists in any of this segmentation's utterances
                    if any([utt["text_coref"] for utt in utterances]):
                        #print("skipping ", audioitem['title'], podcast['title'])
                        continue
                    
                    print("starting episode: ", audioitem['title'], podcast['title'])
                    nlp = spacy.load("en_core_web_lg")
                    nlp.add_pipe(
                        "fastcoref", 
                        config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
                    )

                    # add a context field to each utterance with the 50 previous utterances
                    for i, utt in enumerate(utterances):
                        utt["context"] = " ".join([u["text"] for u in utterances[max(0, i-30):i+1]])

                    docs = nlp.pipe(
                    [utterance["context"] for utterance in utterances],
                    component_cfg={"fastcoref": {'resolve_text': True}}
                    )
                    #docs = list(docs)
                    try:
                        docs = list(docs)
                    except:
                        print("error with ", audioitem['title'], podcast['title'])
                        for doc in docs:
                            print(doc)
                            print(utterances)
                        continue

                    assert len(docs) == len(utterances)

                    for i, doc in enumerate(docs):
                        resolved_text = doc._.resolved_text

                        # Create a new Doc object without running the entire pipeline
                        sentences_doc = nlp.make_doc(resolved_text)

                        # Apply the "senter" component to the sentences_doc
                        nlp.get_pipe("senter")(sentences_doc)

                        first = coref = next(sentences_doc.sents)
                        for coref in sentences_doc.sents: pass

                        first = original = next(doc.sents)
                        for original in doc.sents: pass
                        if coref.text.strip() != original.text.strip():
                            # write to API
                            utt_uuid = utterances[i]["uuid"]
                            utterances[i]["text_coref"] = coref.text.strip()
                            # no changes to these child record, so remove them
                            utterances[i].pop("classification_set")
                            utterances[i].pop("query_set")
                            res = requests.post(f"{SERVER}:{PORT}/api/utterances/{utt_uuid}/", json=utterances[i])
                            print(res.status_code)

                    

starting episode:  Ep. 1714 - Corpse Declares Re-election Launch The Ben Shapiro Show


05/18/2023 01:24:58 - INFO - 	 missing_keys: []
05/18/2023 01:24:58 - INFO - 	 unexpected_keys: []
05/18/2023 01:24:58 - INFO - 	 mismatched_keys: []
05/18/2023 01:24:58 - INFO - 	 error_msgs: []
05/18/2023 01:24:58 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
05/18/2023 01:25:03 - INFO - 	 Tokenize 256 inputs...
05/18/2023 01:25:06 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:07<00:00, 32.16it/s]
05/18/2023 01:25:21 - INFO - 	 Tokenize 256 inputs...
05/18/2023 01:25:25 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:13<00:00, 18.46it/s]
05/18/2023 01:25:46 - INFO - 	 Tokenize 256 inputs...
05/18/2023 01:25:49 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:12<00:00, 20.72it/s]


error with  Ep. 1714 - Corpse Declares Re-election Launch The Ben Shapiro Show
starting episode:  Ep. 1708 - The Real Interracial Crime Problem Isn't White On Black, It's Black On White The Ben Shapiro Show


05/18/2023 01:26:07 - INFO - 	 missing_keys: []
05/18/2023 01:26:07 - INFO - 	 unexpected_keys: []
05/18/2023 01:26:07 - INFO - 	 mismatched_keys: []
05/18/2023 01:26:07 - INFO - 	 error_msgs: []
05/18/2023 01:26:07 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
05/18/2023 01:26:15 - INFO - 	 Tokenize 256 inputs...
05/18/2023 01:26:21 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:14<00:00, 17.44it/s]
05/18/2023 01:26:43 - INFO - 	 Tokenize 256 inputs...
05/18/2023 01:26:47 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:14<00:00, 17.82it/s]


error with  Ep. 1708 - The Real Interracial Crime Problem Isn't White On Black, It's Black On White The Ben Shapiro Show
starting episode:  Ep. 1709 -  Fox News Drops Shocking Amount of Money In Defamation Settlement The Ben Shapiro Show


...

05/18/2023 01:49:57 - INFO - 	 missing_keys: []
05/18/2023 01:49:57 - INFO - 	 unexpected_keys: []
05/18/2023 01:49:57 - INFO - 	 mismatched_keys: []
05/18/2023 01:49:57 - INFO - 	 error_msgs: []
05/18/2023 01:49:57 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
05/18/2023 01:50:03 - INFO - 	 Tokenize 247 inputs...
05/18/2023 01:50:07 - INFO - 	 ***** Running Inference on 247 texts *****
Inference: 100%|██████████| 247/247 [00:12<00:00, 19.68it/s]


error with  The Indictment of Donald Trump The Daily
starting episode:  The Day Title 42 Ended The Daily


05/18/2023 01:50:35 - INFO - 	 missing_keys: []
05/18/2023 01:50:35 - INFO - 	 unexpected_keys: []
05/18/2023 01:50:35 - INFO - 	 mismatched_keys: []
05/18/2023 01:50:35 - INFO - 	 error_msgs: []
05/18/2023 01:50:35 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
05/18/2023 01:50:42 - INFO - 	 Tokenize 256 inputs...
05/18/2023 01:50:45 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:13<00:00, 19.29it/s]
05/18/2023 01:50:59 - INFO - 	 Tokenize 12 inputs...
05/18/2023 01:51:00 - INFO - 	 ***** Running Inference on 12 texts *****
Inference: 100%|██████████| 12/12 [00:00<00:00, 15.99it/s]


200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
starting episode:  The Lifesaving Power of … Paperwork? The Daily


05/18/2023 01:51:09 - INFO - 	 missing_keys: []
05/18/2023 01:51:09 - INFO - 	 unexpected_keys: []
05/18/2023 01:51:09 - INFO - 	 mismatched_keys: []
05/18/2023 01:51:09 - INFO - 	 error_msgs: []
05/18/2023 01:51:09 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
05/18/2023 01:51:18 - INFO - 	 Tokenize 256 inputs...
05/18/2023 01:51:23 - INFO - 	 ***** Running Inference on 256 texts *****
Inference:  46%|████▌     | 118/256 [00:07<00:09, 14.50it/s]

In [13]:
nlp = spacy.load("en_core_web_lg", exclude=["parser", "lemmatizer", "ner", "textcat"])
nlp.add_pipe(
    "fastcoref", 
    config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
)
docs = nlp.pipe(
[utterance["context"] for utterance in utterances],
component_cfg={"fastcoref": {'resolve_text': True}}
)

05/13/2023 15:27:23 - INFO - 	 missing_keys: []
05/13/2023 15:27:23 - INFO - 	 unexpected_keys: []
05/13/2023 15:27:23 - INFO - 	 mismatched_keys: []
05/13/2023 15:27:23 - INFO - 	 error_msgs: []
05/13/2023 15:27:23 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M


In [14]:
for doc in docs:
    try:
        print(doc)
    except:
        print("!!!!!!!!!!!!!!!!!!!!!error")
        print(doc)

05/13/2023 15:27:29 - INFO - 	 Tokenize 256 inputs...
05/13/2023 15:27:32 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:12<00:00, 20.95it/s]

Welcome to Washington, D.C. Senator, nice to be here with you.
Welcome to Washington, D.C. Senator, nice to be here with you. Congressman Jim Jordan, our special guest, and I got to say, I know you and I have talked about this on the plane ride up.
Welcome to Washington, D.C. Senator, nice to be here with you. Congressman Jim Jordan, our special guest, and I got to say, I know you and I have talked about this on the plane ride up. This is going to be a really important podcast.
Welcome to Washington, D.C. Senator, nice to be here with you. Congressman Jim Jordan, our special guest, and I got to say, I know you and I have talked about this on the plane ride up. This is going to be a really important podcast. It's going to be a two part series that we're going to do, and we're going to take a big deep breath, start back at the beginning.
Welcome to Washington, D.C. Senator, nice to be here with you. Congressman Jim Jordan, our special guest, and I got to say, I know you and I have talked

TypeError: 'NoneType' object is not subscriptable

In [ ]:
utterances[i].pop("classification_set")

In [ ]:
utterances[i]

In [ ]:
[utterance["context"] for utterance in utterances]

In [ ]:
import copy
for podcast in podcasts:
    if not podcast["language"].startswith("en"):
        continue
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()
                    utterances = seg["utterance_set"]
                    if any([utt["text_coref"] for utt in utterances]):
                        continue
                    
                    print("starting episode: ", audioitem['title'], podcast['title'])
                    nlp = spacy.load("en_core_web_lg")
                    nlp.add_pipe(
                        "fastcoref", 
                        config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
                    )

                    for i, utt in enumerate(utterances):
                        utt["context"] = " ".join([u["text"] for u in utterances[max(0, i-40):i+1]])

                    for utterance in utterances:
                        try:
                            doc = nlp(utterance["context"], component_cfg={"fastcoref": {'resolve_text': True}})
                        except Exception as e:
                            print(f"Error occurred during processing podcast {podcast['title']} with episode {audioitem['title']} and utterance {utterance['context']}:")
                            print(e)
                            continue

                        resolved_text = doc._.resolved_text
                        sentences_doc = nlp.make_doc(resolved_text)
                        nlp.get_pipe("senter")(sentences_doc)

                        first = coref = next(sentences_doc.sents)
                        for coref in sentences_doc.sents: pass

                        first = original = next(doc.sents)
                        for original in doc.sents: pass

                        if coref.text.strip() != original.text.strip():
                            utt_uuid = utterance["uuid"]

                            # Deep copy the utterance
                            utterance_copy = copy.deepcopy(utterance)

                            utterance_copy["text_coref"] = coref.text.strip()
                            if utterance_copy.get("classification_set"):
                                utterance_copy.pop("classification_set")
                            if utterance_copy.get("query_set"):
                                utterance_copy.pop("query_set")
                                                    
                            print(utterance_copy)
                            res = requests.post(f"{SERVER}:{PORT}/api/utterances/{utt_uuid}/", json=utterance_copy)
                            print(res.status_code)


In [ ]:
import spacy
from fastcoref import spacy_component
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()
for i, p in enumerate(podcasts):
    print(f"{i}: {p['title']}")

In [15]:
podcast = podcasts[0]
audioitem = [a for a in podcast['audioitem_set'] if a['guid'] == "01c17681-f1d1-4b4e-bac8-b000000ec0f8"][0]
my_seg = "0ac504bc-f0bc-11ed-a5e2-00155d08852a"
seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{my_seg}/")
seg = seg.json()
utterances = seg["utterance_set"]


print("starting episode: ", audioitem['title'], podcast['title'])
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe(
    "fastcoref", 
    config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
)


In [ ]:

for i, utt in enumerate(utterances):
    utt["context"] = " ".join([u["text"] for u in utterances[max(0, i-40):i+1]])

for utterance in utterances:
    try:
        doc = nlp(utterance["context"], component_cfg={"fastcoref": {'resolve_text': True}})
    except Exception as e:
        print(f"Error occurred during processing podcast {podcast['title']} with episode {audioitem['title']} and utterance {utterance['context']}:")
        print(e)
        continue

    resolved_text = doc._.resolved_text
    sentences_doc = nlp.make_doc(resolved_text)
    nlp.get_pipe("senter")(sentences_doc)

    first = coref = next(sentences_doc.sents)
    for coref in sentences_doc.sents: pass

    first = original = next(doc.sents)
    for original in doc.sents: pass